# Baseline Experiments: TF-IDF + Logistic Regression
## Cross-Dataset Generalization in Twi Sentiment Classification

This notebook runs the four planned experiments (E1–E4) using one fixed baseline model and leakage-controlled evaluation sets.

**Random state:** 42  
**Labels:** positive, negative, neutral

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

ROOT = Path('..')
PROCESSED = ROOT / '03_Data' / 'processed'
RESULTS = ROOT / '05_Results'
FIGURES = RESULTS / 'figures'
TABLES = RESULTS / 'tables'
for p in [RESULTS, FIGURES, TABLES]:
    p.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
LABELS = ['negative', 'neutral', 'positive']

In [2]:
afri = pd.read_csv(PROCESSED / 'afrisenti_twi.csv')
asante = pd.read_csv(PROCESSED / 'asantetwi_senti.csv')
print('AfriSenti:', afri.shape)
print('AsanteTwiSenti:', asante.shape)
print('\nAfri labels:\n', afri['label'].value_counts())
print('\nAsante labels:\n', asante['label'].value_counts())

AfriSenti: (5914, 2)
AsanteTwiSenti: (4818, 2)

Afri labels:
 label
positive    3542
neutral     1308
negative    1064
Name: count, dtype: int64

Asante labels:
 label
positive    2277
negative    1815
neutral      726
Name: count, dtype: int64


In [3]:
# Reproducible splits
afri_train_val, afri_test = train_test_split(afri, test_size=0.15, random_state=RANDOM_STATE, stratify=afri['label'])
afri_train, afri_val = train_test_split(afri_train_val, test_size=0.1765, random_state=RANDOM_STATE, stratify=afri_train_val['label'])

asante_train, asante_rest = train_test_split(asante, test_size=(388+949)/len(asante), random_state=RANDOM_STATE, stratify=asante['label'])
asante_val, asante_test = train_test_split(asante_rest, test_size=949/(388+949), random_state=RANDOM_STATE, stratify=asante_rest['label'])

def leakage_control(train_df, test_df):
    train_texts = set(train_df['text'])
    return test_df[~test_df['text'].isin(train_texts)].reset_index(drop=True)

afri_test_eval = leakage_control(afri_train, afri_test)
asante_test_eval = leakage_control(asante_train, asante_test)

print('Afri split:', len(afri_train), len(afri_val), len(afri_test), '-> evaluation test:', len(afri_test_eval))
print('Asante split:', len(asante_train), len(asante_val), len(asante_test), '-> evaluation test:', len(asante_test_eval))

Afri split: 4138 888 888 -> evaluation test: 782
Asante split: 3481 387 950 -> evaluation test: 762


In [4]:
def make_model():
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95, lowercase=True, strip_accents=None)
    classifier = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')
    return vectorizer, classifier

def run_experiment(name, train_df, test_df, train_name, test_name):
    vectorizer, classifier = make_model()
    X_train = vectorizer.fit_transform(train_df['text'])
    X_test = vectorizer.transform(test_df['text'])
    classifier.fit(X_train, train_df['label'])
    y_true = test_df['label'].to_numpy()
    y_pred = classifier.predict(X_test)

    metrics = {
        'experiment': name, 'train_dataset': train_name, 'test_dataset': test_name,
        'train_size': len(train_df), 'test_size': len(test_df),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }

    print('\n', name, train_name, '→', test_name)
    print(pd.Series(metrics))
    print('\nClassification report:\n', classification_report(y_true, y_pred, labels=LABELS, zero_division=0))

    with open(RESULTS / f'{name}_results.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    (TABLES / f'{name}_classification_report.txt').write_text(classification_report(y_true, y_pred, labels=LABELS, zero_division=0), encoding='utf-8')

    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(cm)
    ax.set_xticks(range(3), ['Negative','Neutral','Positive'])
    ax.set_yticks(range(3), ['Negative','Neutral','Positive'])
    ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label'); ax.set_title(f'{name}: {train_name} → {test_name}')
    for i in range(3):
        for j in range(3): ax.text(j, i, str(cm[i, j]), ha='center', va='center')
    fig.tight_layout(); fig.savefig(FIGURES / f'{name}_confusion_matrix.png', dpi=300, bbox_inches='tight'); plt.close(fig)
    return metrics

## E1 — AfriSenti → AfriSenti
Within-dataset baseline.

In [5]:
e1 = run_experiment('E1', afri_train, afri_test_eval, 'AfriSenti', 'AfriSenti')


 E1 AfriSenti → AfriSenti
experiment                E1
train_dataset      AfriSenti
test_dataset       AfriSenti
train_size              4138
test_size                782
accuracy            0.707161
precision_macro     0.574599
recall_macro        0.573279
f1_macro            0.573934
f1_weighted         0.706666
dtype: object

Classification report:
               precision    recall  f1-score   support

    negative       0.45      0.45      0.45       118
     neutral       0.44      0.44      0.44       135
    positive       0.83      0.83      0.83       529

    accuracy                           0.71       782
   macro avg       0.57      0.57      0.57       782
weighted avg       0.71      0.71      0.71       782



## E2 — AfriSenti → AsanteTwiSenti
Cross-dataset generalization.

In [6]:
e2 = run_experiment('E2', afri_train, asante_test_eval, 'AfriSenti', 'AsanteTwiSenti')


 E2 AfriSenti → AsanteTwiSenti
experiment                     E2
train_dataset           AfriSenti
test_dataset       AsanteTwiSenti
train_size                   4138
test_size                     762
accuracy                 0.481627
precision_macro           0.44228
recall_macro             0.444354
f1_macro                 0.440933
f1_weighted               0.48926
dtype: object

Classification report:
               precision    recall  f1-score   support

    negative       0.50      0.47      0.49       281
     neutral       0.23      0.31      0.27       125
    positive       0.59      0.55      0.57       356

    accuracy                           0.48       762
   macro avg       0.44      0.44      0.44       762
weighted avg       0.50      0.48      0.49       762



## E3 — AsanteTwiSenti → AsanteTwiSenti
Within-dataset baseline.

In [7]:
e3 = run_experiment('E3', asante_train, asante_test_eval, 'AsanteTwiSenti', 'AsanteTwiSenti')


 E3 AsanteTwiSenti → AsanteTwiSenti
experiment                     E3
train_dataset      AsanteTwiSenti
test_dataset       AsanteTwiSenti
train_size                   3481
test_size                     762
accuracy                 0.586614
precision_macro          0.537924
recall_macro             0.534948
f1_macro                 0.533872
f1_weighted              0.596484
dtype: object

Classification report:
               precision    recall  f1-score   support

    negative       0.69      0.62      0.65       281
     neutral       0.25      0.34      0.29       125
    positive       0.67      0.65      0.66       356

    accuracy                           0.59       762
   macro avg       0.54      0.53      0.53       762
weighted avg       0.61      0.59      0.60       762



## E4 — AsanteTwiSenti → AfriSenti
Reverse cross-dataset generalization.

In [8]:
e4 = run_experiment('E4', asante_train, afri_test_eval, 'AsanteTwiSenti', 'AfriSenti')


 E4 AsanteTwiSenti → AfriSenti
experiment                     E4
train_dataset      AsanteTwiSenti
test_dataset            AfriSenti
train_size                   3481
test_size                     782
accuracy                 0.516624
precision_macro          0.413418
recall_macro             0.437806
f1_macro                 0.412782
f1_weighted              0.543939
dtype: object

Classification report:
               precision    recall  f1-score   support

    negative       0.28      0.47      0.35       118
     neutral       0.20      0.25      0.22       135
    positive       0.77      0.60      0.67       529

    accuracy                           0.52       782
   macro avg       0.41      0.44      0.41       782
weighted avg       0.59      0.52      0.54       782



## Main comparison
The main outcome is the change in Macro F1 between within-dataset and cross-dataset evaluation.

In [9]:
results = pd.DataFrame([e1, e2, e3, e4])
results['setting'] = ['within', 'cross', 'within', 'cross']
results['generalization_gap'] = np.nan
results.loc[results['experiment']=='E2', 'generalization_gap'] = e1['f1_macro'] - e2['f1_macro']
results.loc[results['experiment']=='E4', 'generalization_gap'] = e3['f1_macro'] - e4['f1_macro']
results.to_csv(RESULTS / 'main_results.csv', index=False)
within_avg = (e1['f1_macro'] + e3['f1_macro']) / 2
cross_avg = (e2['f1_macro'] + e4['f1_macro']) / 2
print(results[['experiment','train_dataset','test_dataset','accuracy','f1_macro','f1_weighted','generalization_gap']])
print(f'\nAverage within-dataset Macro F1: {within_avg:.4f}')
print(f'Average cross-dataset Macro F1:  {cross_avg:.4f}')
print(f'Average generalization gap:      {within_avg-cross_avg:.4f}')
(RESULTS / 'generalization_summary.json').write_text(json.dumps({'average_within_macro_f1': within_avg, 'average_cross_macro_f1': cross_avg, 'average_generalization_gap': within_avg-cross_avg}, indent=2))

  experiment   train_dataset  ... f1_weighted  generalization_gap
0         E1       AfriSenti  ...    0.706666                 NaN
1         E2       AfriSenti  ...    0.489260            0.133001
2         E3  AsanteTwiSenti  ...    0.596484                 NaN
3         E4  AsanteTwiSenti  ...    0.543939            0.121090

[4 rows x 7 columns]

Average within-dataset Macro F1: 0.5539
Average cross-dataset Macro F1:  0.4269
Average generalization gap:      0.1270


152